The Problem
While the baseline model achieved 80% accuracy, the Recall (0.55) was too low for a risk detective tool—we were missing 45% of churners.

The Theory
I identified Multicollinearity between TotalCharges and Tenure, and realized the Class Imbalance (73% vs 27%) was biasing the model's sensitivity.

In [10]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

load_dotenv()
engine = create_engine(os.getenv("SUPABASE_DB_URL"))

def load_cleaned():
    return pd.read_sql("SELECT * FROM telco_customer.stg_churn_cleaned", engine)

# 1. Load the data cleaned in the EDA
df = load_cleaned()

# 2. Separate the "Answer" (y) from the "Clues" (X)
# Drop customer_id (useless) and gender/phone_service (from our EDA findings)
X = df.drop(columns=['churn', 'customer_id', 'gender', 'phone_service'])
y = df['churn']

# 3. Randomly extract 20% for the "Final Exam"
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"We are training on {X_train.shape[0]} customers.")
print(f"We are testing on {X_test.shape[0]} customers.")

We are training on 5625 customers.
We are testing on 1407 customers.


In [11]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Identify which columns need which treatment
# We'll use the columns currently in your X_train
numeric_features = ['tenure', 'monthly_charges', 'total_charges']
categorical_features = [col for col in X_train.columns if col not in numeric_features]

# 2. Define the transformation 'rules'
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

print("Preprocessing rules defined!")

Preprocessing rules defined!


In [12]:
# --- Phase 2: Optimized Model (Handling Imbalance & Redundancy) ---

# 1. Update the feature lists to EXCLUDE total_charges
numeric_features_opt = ['tenure', 'monthly_charges']
categorical_features_opt = [col for col in X_train.columns if col not in numeric_features_opt + ['total_charges']]

# 2. Re-define the preprocessor for THIS specific model
preprocessor_opt = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_opt),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features_opt)
    ])

# 3. Create the Optimized Pipeline
model_optimized = Pipeline(steps=[
    ('preprocessor', preprocessor_opt),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

# 4. Drop the column from your training data BEFORE fitting
X_train_opt = X_train.drop(columns=['total_charges'])
X_test_opt = X_test.drop(columns=['total_charges'])

# 5. Fit the model
model_optimized.fit(X_train_opt, y_train)

# 6. Evaluate
y_pred_opt = model_optimized.predict(X_test_opt)
print("--- Optimized Classification Report ---")
print(classification_report(y_test, y_pred_opt))

--- Optimized Classification Report ---
              precision    recall  f1-score   support

       False       0.89      0.72      0.79      1033
        True       0.49      0.75      0.59       374

    accuracy                           0.72      1407
   macro avg       0.69      0.73      0.69      1407
weighted avg       0.78      0.72      0.74      1407



In [13]:
# Extract the new coefficients
feat_names_opt = model_optimized.named_steps['preprocessor'].get_feature_names_out()
weights_opt = model_optimized.named_steps['classifier'].coef_[0]

coef_df_opt = pd.DataFrame({'Feature': feat_names_opt, 'Weight': weights_opt}).sort_values(by='Weight', ascending=False)

print("\n--- Optimized Top Churn Drivers ---")
print(coef_df_opt.head(5))
print("\n--- Optimized Top Retention Anchors ---")
print(coef_df_opt.tail(5))


--- Optimized Top Churn Drivers ---
                                 Feature    Weight
7      cat__internet_service_Fiber optic  0.978135
5   cat__multiple_lines_No phone service  0.451867
25  cat__payment_method_Electronic check  0.406443
20             cat__streaming_movies_Yes  0.371314
6                cat__multiple_lines_Yes  0.315227

--- Optimized Top Retention Anchors ---
                     Feature    Weight
16     cat__tech_support_Yes -0.344044
10  cat__online_security_Yes -0.367146
21    cat__contract_One year -0.688018
0                num__tenure -0.830535
22    cat__contract_Two year -1.340061


In [16]:
# Side-by-side comparison: Baseline vs Optimised
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pandas as pd

# Re-run baseline predictions (model_pipeline from notebook 2 logic)
model_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])
model_baseline.fit(X_train, y_train)
y_pred_baseline = model_baseline.predict(X_test)

# Optimised predictions (already fitted above)
# Use the same test frame so the column layout matches what the fitted pipeline expects.
y_pred_opt = model_optimized.predict(X_test)

# Build comparison table
def extract_metrics(y_true, y_pred, label):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    churn_key = 'True' if 'True' in report else ('1' if '1' in report else None)
    churn_metrics = report[churn_key] if churn_key else {'precision': 0, 'recall': 0, 'f1-score': 0}
    return {
        'Model': label,
        'Accuracy': round(report['accuracy'], 3),
        'Precision (Churn)': round(churn_metrics['precision'], 3),
        'Recall (Churn)': round(churn_metrics['recall'], 3),
        'F1 (Churn)': round(churn_metrics['f1-score'], 3),
    }

comparison = pd.DataFrame([
    extract_metrics(y_test, y_pred_baseline, 'Baseline'),
    extract_metrics(y_test, y_pred_opt,      'Optimised'),
])
comparison.set_index('Model')

# Recall (Churn) is the key metric here — we care more about catching churners
# than overall accuracy. The optimised model trades some accuracy for better sensitivity.

,Accuracy,Precision (Churn),Recall (Churn),F1 (Churn)
Model,,,,
Baseline,0.796,0.634,0.551,0.589
Optimised,0.726,0.490,0.751,0.593


# Summary:

- The Sensitivity Trade-off: By optimizing for Recall, we successfully increased our detection rate from 55% to 75%. While this reduced overall accuracy and precision (resulting in more "false alarms"), the business cost of a false alarm (e.g., sending a discount to a loyal customer) is significantly lower than the cost of losing a customer entirely.

- Verified Churn Drivers: With the removal of redundant data, Fiber Optic service (Weight: 0.98) and Electronic Check payments (Weight: 0.41) were confirmed as the primary triggers for customer exit.

- The "Lock-In" Anchor: Two-Year Contracts emerged as the single most powerful retention tool in the company's arsenal with a massive negative weight of -1.34.